# 06 — Feature: Memory
Parses the `Memory` string (e.g., `256GB SSD + 1TB HDD`) into structured numeric features.

**Output features:** `Mem_SSD_GB`, `Mem_HDD_GB`, `Mem_Flash_GB`, `Mem_eMMC_GB`, `Mem_Total_GB`, `Mem_HasSSD`, `Mem_IsCombo`, `Mem_IsSingleStorage`, `Mem_IsHybridStorage`, `Mem_PrimaryType_TE`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
ld = pd.read_csv("laptop_data_features.csv")
print(f"Shape: {ld.shape}")

In [ ]:
# Explore Memory
print(ld['Memory'].value_counts().head(20))
print(f"\nUnique Memory configs: {ld['Memory'].nunique()}")

In [ ]:
def parse_size_gb(size_str):
    """Convert '500GB' or '1TB' to integer GB."""
    size_str = size_str.strip()
    match = re.match(r'(\d+\.?\d*)\s*(GB|TB)', size_str, re.IGNORECASE)
    if not match:
        return 0
    val, unit = float(match.group(1)), match.group(2).upper()
    return int(val * 1024) if unit == 'TB' else int(val)

def parse_memory(mem_str):
    """
    Parses memory string into storage breakdown by type.
    Handles single and combo drives (e.g., '256GB SSD +  1TB HDD').
    """
    if pd.isna(mem_str):
        return pd.Series([0, 0, 0, 0, 'Unknown'],
                         index=['Mem_SSD_GB', 'Mem_HDD_GB', 'Mem_Flash_GB', 'Mem_eMMC_GB', 'Mem_PrimaryType'])
    ssd_gb = hdd_gb = flash_gb = emmc_gb = 0
    for part in mem_str.split('+'):
        part = part.strip()
        size_match = re.match(r'(\d+\.?\d*\s*(?:GB|TB))', part, re.IGNORECASE)
        size = parse_size_gb(size_match.group(1)) if size_match else 0
        pl = part.lower()
        if 'ssd' in pl:
            ssd_gb += size
        elif 'hdd' in pl or 'hard disk' in pl:
            hdd_gb += size
        elif 'flash' in pl:
            flash_gb += size
        elif 'emmc' in pl:
            emmc_gb += size
        else:
            ssd_gb += size   # fallback (e.g., bare "128GB")
    type_scores = {'SSD': ssd_gb, 'HDD': hdd_gb, 'Flash': flash_gb, 'eMMC': emmc_gb}
    primary = max(type_scores, key=type_scores.get) if any(type_scores.values()) else 'Unknown'
    return pd.Series([ssd_gb, hdd_gb, flash_gb, emmc_gb, primary],
                     index=['Mem_SSD_GB', 'Mem_HDD_GB', 'Mem_Flash_GB', 'Mem_eMMC_GB', 'Mem_PrimaryType'])

ld[['Mem_SSD_GB', 'Mem_HDD_GB', 'Mem_Flash_GB', 'Mem_eMMC_GB', 'Mem_PrimaryType']] = ld['Memory'].apply(parse_memory)
ld['Mem_Total_GB'] = ld['Mem_SSD_GB'] + ld['Mem_HDD_GB'] + ld['Mem_Flash_GB'] + ld['Mem_eMMC_GB']
ld['Mem_HasSSD']   = (ld['Mem_SSD_GB'] > 0).astype(int)
ld['Mem_IsCombo']  = ((ld['Mem_SSD_GB'] > 0) & (ld['Mem_HDD_GB'] > 0)).astype(int)

# Count how many distinct storage types are present
ld['_storage_type_count'] = (
    (ld['Mem_SSD_GB']   > 0).astype(int) +
    (ld['Mem_HDD_GB']   > 0).astype(int) +
    (ld['Mem_Flash_GB'] > 0).astype(int) +
    (ld['Mem_eMMC_GB']  > 0).astype(int)
)
ld['Mem_IsSingleStorage'] = (ld['_storage_type_count'] == 1).astype(int)
ld['Mem_IsHybridStorage'] = (ld['_storage_type_count'] >= 2).astype(int)
ld.drop(columns=['_storage_type_count'], inplace=True)

print("Parse sample:")
print(ld[['Memory', 'Mem_SSD_GB', 'Mem_HDD_GB', 'Mem_Total_GB', 'Mem_PrimaryType',
          'Mem_IsSingleStorage', 'Mem_IsHybridStorage']].head(10).to_string())

In [ ]:
# Visualize primary type vs Price
plt.figure(figsize=(8, 4))
sns.barplot(data=ld, x='Mem_PrimaryType', y='Price', estimator='mean',
            order=ld.groupby('Mem_PrimaryType')['Price'].mean().sort_values(ascending=False).index)
plt.title("Mean Price by Primary Storage Type")
plt.xlabel("Storage Type")
plt.ylabel("Mean Price (INR)")
plt.tight_layout()
plt.show()

In [ ]:
# SSD size vs Price
plt.figure(figsize=(7, 4))
plt.scatter(ld['Mem_SSD_GB'], ld['Price'], alpha=0.4, color='steelblue')
plt.title("SSD Capacity vs Price")
plt.xlabel("SSD (GB)")
plt.ylabel("Price (INR)")
plt.tight_layout()
plt.show()

In [ ]:
# HasSSD vs Price (boxplot)
plt.figure(figsize=(6, 4))
sns.boxplot(data=ld, x='Mem_HasSSD', y='Price')
plt.title("Price: Has SSD vs No SSD")
plt.xticks([0, 1], ['No SSD', 'Has SSD'])
plt.tight_layout()
plt.show()

In [ ]:
# Target encode primary type
pt_mean = ld.groupby('Mem_PrimaryType')['Price'].mean()
ld['Mem_PrimaryType_TE'] = ld['Mem_PrimaryType'].map(pt_mean)

In [ ]:
# Correlation with Price
mem_cols = ['Mem_SSD_GB', 'Mem_HDD_GB', 'Mem_Total_GB', 'Mem_HasSSD', 'Mem_IsCombo',
            'Mem_IsSingleStorage', 'Mem_IsHybridStorage', 'Mem_PrimaryType_TE']
print("Correlation with Price:")
print(ld[mem_cols + ['Price']].corr()['Price'].sort_values(ascending=False))

In [ ]:
# Drop raw and intermediate columns
ld.drop(columns=['Memory', 'Mem_PrimaryType'], inplace=True)

In [ ]:
ld.to_csv("laptop_data_features.csv", index=False)
print("Saved: laptop_data_features.csv")
print(f"Shape: {ld.shape}")
print(f"Columns: {list(ld.columns)}")